# 03.4 — Structure preservation

You can read all fifteen documents now, and the text is clean.

But a document is not a bag of sentences. It has sections, and the sections say
what the sentences are about. It has tables, and the header row says what the
numbers mean. It has speakers, and the attribution says whose opinion you're
reading.

Flatten it to plain text and all of that is still technically present — sitting
in a different chunk from the sentence that needed it.

In [1]:
!pip install -q pymupdf4llm==1.28.2

## The problem, stated precisely

Take the chunk that answers our most-asked question.

In [2]:
import pymupdf4llm
from pathlib import Path

CORPUS = Path('../../corpus/docs')

text = pymupdf4llm.to_markdown(str(CORPUS / 'sahel-employee-handbook-2025.pdf'))

i = text.find('25 working days')
print(text[i - 60:i + 200])


## **4. Annual leave** 

Confirmed staff are entitled to **25 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10 working days may be c


Now read it as a retriever would, with no idea where it came from:

> *Confirmed staff are entitled to 25 working days of paid annual leave each
> calendar year...*

Three things that sentence does not say:

- which organisation this is
- that it's the section on annual leave
- that it's the 2025 edition, not the 2023 one

That third one is the failure that cost seven questions in module 02. The chunk
is correct, complete and unattributable.

## Headings are the cheapest context you'll ever get

The parser preserved them. Look.

In [3]:
for line in text.splitlines():
    if line.strip().startswith('#'):
        print(line.strip())

# **Sahel Microfinance Bank Plc**
## **1. Introduction**
## **2. Employment categories**
## **3. Working hours and attendance**
## **4. Annual leave**
## **5. Other leave**
## **6. Expenses and allowances**
## **7. Remote and hybrid work**
## **8. Conduct and discipline**
## **9. Confidentiality and data protection**
## **10. Exit**
## **11. Review**


Every section is marked, and the markers are nested — `#` for the document title,
`##` for each section.

That means you can walk the document keeping track of which heading you're under,
and hand every block of text the path that leads to it.

In [4]:
import re

def sections(markdown):
    """Split markdown into (heading path, body) pairs."""
    stack, out, buf = {}, [], []

    def flush():
        body = '\n'.join(buf).strip()
        if body:
            path = ' > '.join(stack[k] for k in sorted(stack))
            out.append((path, body))
        buf.clear()

    for line in markdown.splitlines():
        heading = re.match(r'^(#{1,6})\s+(.*)$', line.strip())
        if heading:
            flush()
            level = len(heading.group(1))
            title = re.sub(r'[*_`]|<[^>]+>', '', heading.group(2)).strip()
            stack[level] = title
            for deeper in [k for k in stack if k > level]:
                del stack[deeper] # a new h2 invalidates the old h3
        else:
            buf.append(line)

    flush()
    return out

for path, body in sections(text):
    print(f'{len(body):>5} chars  {path}')

   32 chars  
  129 chars  Sahel Microfinance Bank Plc
  579 chars  Sahel Microfinance Bank Plc > 1. Introduction
  522 chars  Sahel Microfinance Bank Plc > 2. Employment categories
  481 chars  Sahel Microfinance Bank Plc > 3. Working hours and attendance
  682 chars  Sahel Microfinance Bank Plc > 4. Annual leave
  819 chars  Sahel Microfinance Bank Plc > 5. Other leave
  803 chars  Sahel Microfinance Bank Plc > 6. Expenses and allowances
  427 chars  Sahel Microfinance Bank Plc > 7. Remote and hybrid work
  591 chars  Sahel Microfinance Bank Plc > 8. Conduct and discipline
  362 chars  Sahel Microfinance Bank Plc > 9. Confidentiality and data protection
  354 chars  Sahel Microfinance Bank Plc > 10. Exit
  362 chars  Sahel Microfinance Bank Plc > 11. Review


Thirteen sections, each carrying its full path.

Two details in that function worth stealing. The `stack` is keyed by heading level
rather than being a list, and a new heading at level *n* deletes everything deeper
than *n* — otherwise a subsection from one part of the document leaks into the
next. And the title is stripped of markdown emphasis and stray HTML tags, because
the parser leaves `**bold**` and the occasional `<u>` in heading text.

Now attach the path to the text:

In [5]:
def with_context(path, body):
    return f'[{path}]\n{body}'

for path, body in sections(text):
    if '25 working days' in body:
        print(with_context(path, body)[:260])

[Sahel Microfinance Bank Plc > 4. Annual leave]
Confirmed staff are entitled to **25 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10


The chunk now says which organisation and which section it comes from.

It still doesn't say which *edition*. That's not a structural property of the
document — it's metadata about the file, and it's notebook 5.

This technique has a name in the literature — prepending context to chunks before
embedding them. It's cheap, it costs a few tokens per chunk, and it's one of the
highest-value things in this module. You'll measure it in notebook 6.

## Attribution: who said that?

The board minutes are a harder case, and a common one — anything derived from a
meeting, an interview or a chat log has the same shape.

In [6]:
minutes = pymupdf4llm.to_markdown(str(CORPUS / 'kaduna-agro-board-minutes-2024-10-17.pdf'))

for path, body in sections(minutes):
    if 'payback' in body.lower():
        print(f'[{path}]\n')
        print(re.sub(r'\s+', ' ', body)[:340])

[Kaduna Agro Processing Limited > 4. Energy costs and captive generation]

**Mr Adeyemi** presented the feasibility study on captive gas generation at Kaduna. Capital outlay was estimated at NGN 1.9 billion with a payback of thirty-one months at current tariffs. **Mrs Bello** questioned the sensitivity of the payback to gas pricing and asked that the study be re-run at a gas price twenty-five per cent above the 


Read that as three separate sentences, because that's roughly what a chunker will
do to it:

1. Someone presented a feasibility study, capital outlay NGN 1.9 billion, payback
   thirty-one months.
2. Someone questioned the sensitivity of the payback and asked for a re-run.
3. Someone observed the Board shouldn't commit capital yet, and proposed deferring.

Split those across chunks and you can still answer *what was the payback period*.
You cannot answer *who raised the concern*, or *who decided to defer* — which for
board minutes is most of what anyone wants to know.

The names survived parsing as bold text, so they're recoverable:

In [7]:
SPEAKER = re.compile(r'\*\*((?:Mr|Mrs|Dr|Ms|Alhaji|The Chairman)[^*]{0,30})\*\*')

for path, body in sections(minutes):
    speakers = SPEAKER.findall(body)
    if speakers:
        print(f'{path.split(" > ")[-1]:<42} {list(dict.fromkeys(speakers))}')

3. Managing Director's report              ['Mrs Okonkwo', 'Dr Yusuf', 'Ms Etim']
4. Energy costs and captive generation     ['Mr Adeyemi', 'Mrs Bello', 'The Chairman']
5. Haulage contract award                  ['Mr Adeyemi', 'Dr Yusuf']
6. Network consolidation                   ['Mrs Okonkwo', 'Mr Nwosu', 'The Chairman']
7. Any other business                      ['Mr Sani']


Every section, with the people who spoke in it.

There are two ways to use that. Put the speaker list in the section's context line,
so any chunk from that section knows who was involved. Or split the section by
speaker turn, so each chunk contains exactly one person's contribution — which is
better for attribution and worse for the surrounding discussion.

There's no general right answer. For minutes, turn-level usually wins. For a
narrative interview, the discussion matters more than the turn boundaries.

One warning. That regex works because these minutes use a consistent format and
the parser preserved the bold. Both of those are luck. Real transcripts arrive as
`SPEAKER 1:`, or `[00:14:22] Grace:`, or with no markers at all — and a regex that
silently matches nothing gives you a document with no attribution and no error.

## Tables lose their headers

Notebook 1 in module 02 showed this and moved on. It's a structure problem, so it
belongs here.

In [8]:
report = pymupdf4llm.to_markdown(str(CORPUS / 'kaduna-agro-annual-report-2024.pdf'))

rows = [l for l in report.splitlines() if l.strip().startswith('|')]
print(f'{len(rows)} table rows\n')
print('first three:')
for r in rows[:3]:
    print(' ', r)
print('\nrows 20-22, which a chunker may well separate from the header:')
for r in rows[20:23]:
    print(' ', r)

49 table rows

first three:
  |**State**|**Local Government Area**|**Volume (tonnes)**|**Revenue (NGN m)**|**Utilisation**|
  |---|---|---|---|---|
  |Lagos|Ikeja|31,447|18,380.5|95%|

rows 20-22, which a chunker may well separate from the header:
  |Rivers|Obio-Akpor|7,252|4,021.1|61%|
  |Rivers|Bonny|16,851|8,719.5|42%|
  |Delta|Warri|6,402|3,373.8|47%|


A chunk containing rows 20 to 22 is a grid of numbers with no column names.
`|Kano|Dala|34,553|14,542.4|46%|` — is 34,553 the volume or the revenue? Is 46%
utilisation or margin? The chunk cannot say, and neither can a model reading it.

The fix is the same idea as the heading path: carry the header with every piece.

In [9]:
def table_blocks(markdown, rows_per_block=8):
    """Split markdown tables into blocks, repeating the header in each."""
    blocks, header, body = [], None, []

    def flush():
        if header  and body:
            for start in range(0, len(body), rows_per_block):
                blocks.append('\n'.join([header, *body[start:start + rows_per_block]]))

    for line in markdown.splitlines():
        stripped = line.strip()
        if stripped.startswith('|'):
            if set(stripped) <= set('|-: '):
                continue  # the |---|---| separator row
            if header is None:
                header = stripped
            else:
                body.append(stripped)
        elif header:
            flush()
            header, body = None, []
    flush()
    return blocks

blocks = table_blocks(report)
print(f'{len(blocks)} table blocks\n')
print(blocks[0])

6 table blocks

|**State**|**Local Government Area**|**Volume (tonnes)**|**Revenue (NGN m)**|**Utilisation**|
|Lagos|Ikeja|31,447|18,380.5|95%|
|Lagos|Apapa|32,316|16,347.0|95%|
|Lagos|Ikorodu|40,294|18,377.3|92%|
|Lagos|Badagry|35,348|18,041.4|80%|
|Ogun|Abeokuta|14,001|6,130.8|60%|
|Ogun|Sagamu|11,092|4,849.7|92%|


Every row in that block now carries its column names.

But look at a block from later in the same table.

In [10]:
print(blocks[3])

|Ogun|Ijebu-Ode|43,379|18,564.4|66%|
|Enugu|Nsukka|5,229|2,663.1|86%|
|Anambra|Onitsha|27,845|13,915.8|77%|
|Anambra|Awka|43,051|19,705.0|84%|
|Anambra|Nnewi|19,480|9,428.5|60%|
|Kwara|Ilorin West|23,596|9,978.2|67%|
|Kwara|Offa|9,532|4,247.2|86%|
|Plateau|Jos North|8,422|3,554.8|70%|
|Plateau|Barkin Ladi|33,704|15,293.5|76%|


The header is `|Ogun|Ijebu-Ode|43,379|18,564.4|66%|` — a data row.

This is the page-break problem from module 02, arriving again. The table continues
onto a second page, the parser emitted the continuation as a **new** markdown
table, and the first data row of that page became its header. Our function does
exactly what it was told: it repeats the header it was given.

So the technique is correct and the input is wrong, which is the more common
situation than either being wrong on its own.

Two things follow. A repeated header is only as good as the header, so a sanity
check is worth having — if the header row is all digits, it probably isn't a
header. And detecting that a table continues across a page break, then stitching
the fragments back together, is real work that belongs in module 15.

For tables that fit on one page — most of them — what you have here is right.

## What's next

Three kinds of structure recovered: section paths, speaker attribution, table
headers. All of it was in the document; none of it would have survived a naive
flatten.

One thing is still missing, and it's the important one. Nothing here knows the
handbook chunk came from the **2025** edition rather than the 2023 one. That
information was never in the text — it's a property of the file.

Notebook 5 is about metadata, and it's the notebook that repairs the failure that
cost you seven questions.